# Day 5 — Scikit-learn Pipelines & Tuned Mini-Project

Todays mini project builds a complete production-style pipeline that automatically prevents data leakage and reduces model delivery time. Preprocessing engineered features from day 4, hyperparameter tuning with cross validation and a final test evaluation against the baseline.
    

In [77]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report

In [78]:
df = pd.read_csv("train.csv")


age_med=df["Age"].median()
df["Age"]= df["Age"].fillna(age_med)

df['family_size'] = df['SibSp'] + df['Parch'] + 1
df['Title'] = df['Name'].str.extract(r',\s*([^\.]*)\.')
df['Title'] = df['Title'].replace(['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Ref', 'Sir', 'Jonkheer', 'Dona'],'Rare')
df['Title'] = df['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme':'Mrs'})


df = df.drop(columns=["Cabin","Name","Ticket","PassengerId"], errors='ignore')

df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
df["Fare"] = df["Fare"].fillna(df["Fare"].median())

df.dtypes

Survived         int64
Pclass           int64
Sex             object
Age            float64
SibSp            int64
Parch            int64
Fare           float64
Embarked        object
family_size      int64
Title           object
dtype: object

In [79]:
categ_cols = df.select_dtypes(include=['object', 'bool']).columns.tolist()
low_cardinality_categ = [col for col in df.select_dtypes(include=['int64','float64']).columns 
                           if df[col].nunique() <= 5 and col!="Survived"]

categorical_cols= categ_cols+ low_cardinality_categ

numeric_cols = [col for col in df.select_dtypes(include=['int64', 'float64']).columns
                  if col not in categorical_cols and col!="Survived"]

print("Categorical columns:",categorical_cols)
print("Numerical columns:",numeric_cols)

Categorical columns: ['Sex', 'Embarked', 'Title', 'Pclass']
Numerical columns: ['Age', 'SibSp', 'Parch', 'Fare', 'family_size']


In [80]:
X = df.drop(["Survived"], axis = 1)
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

After cleaning, classifying columns and splitting the dataset the next step is building the pipeline.

### Build a Pipeline with ColumnTransformer

In [81]:
pre = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

pipe = Pipeline([("pre", pre),
                 ("model", RandomForestClassifier(random_state=42))])

    Instead of preprocessing numeric and categorical columns seperately, ColumnTransformer is used inside a pipeline;
    Numeric columns - scaled, categorical columns - One-hot encoded. Running the above code ensures that all preprocessing is fit only on traininng data which prevents data leakage.

### Tuning the Pipline with GridSearchCV

In [82]:
param_grid = {"model__n_estimators": [100,200, 400],
              "model__max_depth": [None, 5, 10, 20],
              "model__min_samples_split": [2,5,10],
              "model__min_samples_leaf": [1,2,4]}

grid = GridSearchCV(pipe, param_grid=param_grid, cv=5, scoring="f1", n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

print("Best CV Score:", grid.best_score_)
print("Best parameters:", grid.best_params_)

best_pipe = grid.best_estimator_

Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best CV Score: 0.7618541958660704
Best parameters: {'model__max_depth': None, 'model__min_samples_leaf': 2, 'model__min_samples_split': 2, 'model__n_estimators': 200}


    With preprocessing and Feature Engineering inside a single Pipeline, running GridSearchCV with 5-fold cross-validation tunes the model hyperparameters and gives a more honset estimate of generalization performance than tuning the model alone. The above settings were recored among the best of all combinations.

In [83]:
baseline = DummyClassifier(strategy="most_frequent", random_state=42)

baseline.fit(X_train,y_train)
baseline_pred = baseline.predict(X_test)
baseline_acc = accuracy_score(y_test, baseline_pred)

tuned_pred = best_pipe.predict(X_test)
tuned_acc = accuracy_score(y_test, tuned_pred)

print("Baselined accuracy", baseline_acc) 
print("Tuned accuracy", tuned_acc) 
print("Improvement over baseline", tuned_acc-baseline_acc) 

Baselined accuracy 0.6145251396648045
Tuned accuracy 0.8379888268156425
Improvement over baseline 0.22346368715083798


In [84]:
print("\nClassification Report (Tuned):\n",classification_report(y_test, tuned_pred))


Classification Report (Tuned):
               precision    recall  f1-score   support

           0       0.83      0.92      0.87       110
           1       0.84      0.71      0.77        69

    accuracy                           0.84       179
   macro avg       0.84      0.81      0.82       179
weighted avg       0.84      0.84      0.83       179



In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

ConfusionMatrixDisplay.from_estimator(best_pipe, x_test, y_test, cmap="Blues")
plt.title("Confusion Matrix - Tuned Pipeline")

    The test set was never touched during training/tuning. The tuned pipeline showed an improvement of 0.22 in accuracy. The classification reposrt shows good performance in both (Survived/died) classes with better performance in predicting those who died. The model detects those who die better than those who survive.